# Model Router — Results Analysis

Generates figures for the paper from `experiments/results/*.json` output files.

**Install deps:** `pip install pandas matplotlib seaborn`

In [ ]:
import json
import pathlib
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.2)
FIGURES_DIR = pathlib.Path('analysis/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Load the most recent results file (or specify one explicitly)
results_dir = pathlib.Path('experiments/results')
result_files = sorted(results_dir.glob('results_*.json'))

if not result_files:
    raise FileNotFoundError(
        'No results files found. Run: python experiments/run_eval.py'
    )

result_path = result_files[-1]  # most recent
print(f'Loading: {result_path}')

with open(result_path) as f:
    data = json.load(f)

summary = data['summary']
df = pd.DataFrame(data['results'])
print(f"Loaded {len(df)} results | Accuracy: {summary['routing_accuracy']:.1%} | "
      f"Savings: {summary['savings_pct']:.1f}%")

## Figure 1 — Routing Accuracy by Tier

In [ ]:
tier_order = ['simple', 'moderate', 'complex', 'sensitive']
tier_colors = {'simple': '#4ade80', 'moderate': '#60a5fa', 'complex': '#f59e0b', 'sensitive': '#f87171'}

accuracy_by_tier = (
    df.groupby('expected_tier')['tier_correct']
    .agg(['mean', 'count'])
    .reindex(tier_order)
    .reset_index()
)
accuracy_by_tier.columns = ['tier', 'accuracy', 'count']

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(
    accuracy_by_tier['tier'],
    accuracy_by_tier['accuracy'],
    color=[tier_colors[t] for t in accuracy_by_tier['tier']],
    width=0.5, edgecolor='white', linewidth=1.5
)

# Annotate bars
for bar, acc, n in zip(bars, accuracy_by_tier['accuracy'], accuracy_by_tier['count']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{acc:.0%}\n(n={n})', ha='center', va='bottom', fontsize=10)

ax.set_ylim(0, 1.15)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.set_xlabel('Expected Complexity Tier')
ax.set_ylabel('Routing Accuracy')
ax.set_title('Routing Accuracy by Complexity Tier')
ax.axhline(summary['routing_accuracy'], color='gray', linestyle='--', linewidth=1,
           label=f"Overall: {summary['routing_accuracy']:.1%}")
ax.legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig1_routing_accuracy.pdf', dpi=150)
plt.show()
print('Saved fig1_routing_accuracy.pdf')

## Figure 2 — Cost Savings vs. Baseline

In [ ]:
cost_by_tier = (
    df.groupby('expected_tier')[['cost_usd', 'baseline_cost_usd']]
    .sum()
    .reindex(tier_order)
    .reset_index()
)

x = range(len(tier_order))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar([i - width/2 for i in x], cost_by_tier['baseline_cost_usd'],
       width, label='Baseline (all top-tier)', color='#334155', alpha=0.85)
ax.bar([i + width/2 for i in x], cost_by_tier['cost_usd'],
       width, label='Router (actual)', color='#6366f1', alpha=0.85)

ax.set_xticks(list(x))
ax.set_xticklabels(tier_order)
ax.set_xlabel('Expected Complexity Tier')
ax.set_ylabel('Total Cost (USD)')
ax.set_title(
    f"Cost: Router vs. Baseline  "
    f"(overall savings: {summary['savings_pct']:.1f}%)"
)
ax.legend()
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('$%.5f'))

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig2_cost_savings.pdf', dpi=150)
plt.show()
print('Saved fig2_cost_savings.pdf')

## Figure 3 — Latency Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

for tier in tier_order:
    subset = df[df['expected_tier'] == tier]['latency_ms']
    ax.hist(subset, bins=20, alpha=0.6, label=tier, color=tier_colors[tier], edgecolor='white')

ax.axvline(df['latency_ms'].quantile(0.99), color='red', linestyle='--', linewidth=1.5,
           label=f"p99: {df['latency_ms'].quantile(0.99):.0f}ms")
ax.axvline(df['latency_ms'].mean(), color='gray', linestyle='--', linewidth=1.5,
           label=f"mean: {df['latency_ms'].mean():.0f}ms")

ax.set_xlabel('End-to-End Latency (ms)')
ax.set_ylabel('Count')
ax.set_title('Latency Distribution by Tier')
ax.legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig3_latency_distribution.pdf', dpi=150)
plt.show()
print('Saved fig3_latency_distribution.pdf')

## Figure 4 — Model Usage Distribution

In [ ]:
model_counts = df['routed_model'].value_counts()

fig, ax = plt.subplots(figsize=(6, 6))
wedge_colors = ['#4ade80', '#60a5fa', '#f87171', '#f59e0b', '#a78bfa']
wedges, texts, autotexts = ax.pie(
    model_counts.values,
    labels=model_counts.index,
    autopct='%1.1f%%',
    colors=wedge_colors[:len(model_counts)],
    startangle=140,
    pctdistance=0.8,
)
for at in autotexts:
    at.set_fontsize(10)

ax.set_title('Request Distribution by Routed Model')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig4_model_distribution.pdf', dpi=150)
plt.show()
print('Saved fig4_model_distribution.pdf')

## Table — Per-Tier Summary (copy into paper)

In [ ]:
table = pd.DataFrame([
    {
        'Tier': tier,
        'N': stats['count'],
        'Accuracy': f"{stats['accuracy']:.1%}",
        'Avg Latency (ms)': f"{stats['avg_latency_ms']:.1f}",
        'Cost (USD)': f"${stats['total_cost_usd']:.6f}",
        'Savings (USD)': f"${stats['total_savings_usd']:.6f}",
    }
    for tier, stats in summary['by_tier'].items()
])

print('\n=== Per-Tier Results ===')
print(table.to_string(index=False))
print(f"\nOverall routing accuracy : {summary['routing_accuracy']:.1%}")
print(f"Total savings vs baseline: {summary['savings_pct']:.1f}%")
print(f"p99 latency overhead     : {summary['p99_latency_ms']:.0f} ms")

# LaTeX table for paper
print('\n=== LaTeX ===')
print(table.to_latex(index=False, caption='Per-tier routing results.', label='tab:results'))